# Test the Metropolis package import

This developer example checks that Python can import `metropolis`, uses its runtime configuration helpers, and loads the bundled Metropolis v1 address plan into a pandas table. The plan can be loaded from an installed wheel, without a repository checkout.

Before running, install the project and development dependencies with `uv sync --group dev`, then select that environment as this notebook's Python kernel.

In [ ]:
import metropolis

print(f"Imported metropolis from: {metropolis.__file__}")

In [ ]:
from metropolis.config import RuntimeConfig, resolve_device

runtime = RuntimeConfig()
print(runtime)
print(f"Available runtime device: {resolve_device(runtime.device)}")

## Load the testbed address plan

This treats the topology file as structured dataset metadata. Each row represents one LAN, router transit, WAN underlay, or reserved overlay network.

In [ ]:
import pandas as pd
from metropolis.testbeds import load_address_plan

address_plan = load_address_plan()

networks = pd.DataFrame(address_plan["networks"])
print(f"Testbed: {address_plan['testbed_id']}")
print(f"Networks: {len(networks)}")
display(networks[["id", "site", "zone", "kind", "cidr", "gateway", "vlan_id"]])

In [ ]:
# A few useful summaries for exploring the topology.
display(networks.groupby(["site", "zone"], dropna=False).size().rename("network_count"))
display(
    networks[networks["kind"].isin(["transit", "underlay", "overlay"])][
        ["id", "kind", "cidr", "purpose"]
    ]
)

In [ ]:
# Basic integrity checks that are useful while editing the topology data.
assert address_plan["testbed_id"] == "metropolis"
assert networks["id"].is_unique
assert networks["cidr"].notna().all()
print("Import and address-plan checks passed.")